# Việc B2 — Ghép GAM và GBM theo quãng đường

> **Mentor:** *"Chuyến bình thường ưu tiên GBM, còn chuyến càng dài thì tăng dần ảnh hưởng của GAM.
> Không cần xây thêm kiến trúc phức tạp."*

> ⚠️ **Chỉ chạy notebook này nếu `02_GAM_ON_DINH` cho thấy lợi thế ổn định.** Ghép hai model dựa
> trên một lợi thế không ổn định chỉ làm hệ thống phức tạp thêm mà không được gì.

## Cách ghép

Trộn tuyến tính, trọng số của GAM tăng dần theo quãng đường:

$$\hat p = (1 - \alpha(d)) \cdot \hat p_{GBM} + \alpha(d) \cdot \hat p_{GAM}$$

$$\alpha(d) = \begin{cases}
0 & d \le d_0 \\
\alpha_{max} \cdot \dfrac{d - d_0}{d_1 - d_0} & d_0 < d < d_1 \\
\alpha_{max} & d \ge d_1
\end{cases}$$

Ba tham số: `d₀` (bắt đầu trộn), `d₁` (trộn hết mức), `α_max` (mức trộn tối đa).

## Chống tự lừa mình

Dò tham số rồi báo cáo kết quả trên chính tập đã dò là cách chắc chắn nhất để ra một con số đẹp mà
vô nghĩa. Notebook này tách đôi:

| Tập | Dùng để | Dữ liệu |
|---|---|---|
| **Dò tham số** | Tìm `d₀`, `d₁`, `α_max` | Test tháng **2026-01** |
| **Đánh giá** | Báo cáo con số cuối | Test tháng **2026-02** và **2026-03**, chưa từng đụng tới |

Con số đưa vào báo cáo là con số ở tập đánh giá.

In [ ]:
import warnings, time, sys, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = ("#0072B2", "#E69F00", "#009E73",
                                         "#D55E00", "#CC79A7", "#666666")
INK = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10.5,
})
EVAL = Path("../model/evaluation")
DATA = Path("../data/hcm_train_ready.parquet")
HINH = Path("../docs/hinh_anh"); HINH.mkdir(parents=True, exist_ok=True)
KQ   = Path("ket_qua"); KQ.mkdir(exist_ok=True)

In [ ]:
# ══ NHÓM CỐ ĐỊNH — quy tắc chốt cho cả tuần 5 ══════════════════════════
# Mentor tuần 4: "giữ nguyên nhóm chuyến giữa các model. Không nên để mỗi model
# tự chia nhóm theo giá mà chính nó dự đoán."
# => nhóm chia theo GIÁ THẬT và QUÃNG ĐƯỜNG, hai thứ không phụ thuộc model nào.
CAT_GIA = [0, 50e3, 100e3, 150e3, 200e3, 300e3, np.inf]
TEN_GIA = ["<50k", "50–100k", "100–150k", "150–200k", "200–300k", ">300k"]
CAT_KM  = [0, 2, 5, 8, 12, 15, np.inf]
TEN_KM  = ["<2", "2–5", "5–8", "8–12", "12–15", ">15"]

def gan_nhom(d, cot_gia="y", cot_km="km"):
    d = d.copy()
    d["band"] = pd.cut(d[cot_gia], CAT_GIA, labels=TEN_GIA)
    d["kmb"]  = pd.cut(d[cot_km],  CAT_KM,  labels=TEN_KM)
    return d

def mape(p, y):
    p, y = np.asarray(p, float), np.asarray(y, float)
    return float(np.mean(np.abs(p - y) / y))

def boot_hieu(p_moc, p_moi, y, B=2000, seed=7):
    """CI 95% cho (sai số mốc − sai số mới). Dương = phương án mới TỐT HƠN."""
    y = np.asarray(y, float)
    h = np.abs(np.asarray(p_moc, float) - y)/y - np.abs(np.asarray(p_moi, float) - y)/y
    if len(h) < 2:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    mau = rng.integers(0, len(h), size=(B, len(h)))
    pp = h[mau].mean(axis=1)
    return h.mean(), float(np.percentile(pp, 2.5)), float(np.percentile(pp, 97.5))

def bang_theo_nhom(d, cot_moc, cot_moi, ten_moc="mốc", ten_moi="mới"):
    """MAPE hai phương án trên từng nhóm cố định + CI của chênh lệch."""
    hang = []
    for cot_nhom, nhan in [("kmb", "km"), ("band", "giá thật")]:
        for g, s in d.groupby(cot_nhom, observed=True):
            if len(s) < 30:
                continue
            m, lo, hi = boot_hieu(s[cot_moc], s[cot_moi], s.y)
            hang.append({"Chia theo": nhan, "Nhóm": str(g), "n": len(s),
                         ten_moc: mape(s[cot_moc], s.y), ten_moi: mape(s[cot_moi], s.y),
                         "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                         "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    m, lo, hi = boot_hieu(d[cot_moc], d[cot_moi], d.y)
    hang.append({"Chia theo": "—", "Nhóm": "TOÀN TẬP", "n": len(d),
                 ten_moc: mape(d[cot_moc], d.y), ten_moi: mape(d[cot_moi], d.y),
                 "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                 "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    return pd.DataFrame(hang)

def in_bang(df, cot_mape):
    return df.style.format({**{c: "{:.2%}" for c in cot_mape},
                            "Chênh (điểm)": "{:+.2f}", "CI thấp": "{:+.2f}",
                            "CI cao": "{:+.2f}", "n": "{:,}"}).hide(axis="index")

In [ ]:
# ══ Nạp dự đoán đã có: Hybrid (mốc) và GAM ═════════════════════════════
ut  = pd.read_parquet(EVAL / "uq_pred_test.parquet").reset_index(drop=True)
gam = pd.read_parquet(EVAL / "pred_gam.parquet").reset_index(drop=True)
assert len(ut) == len(gam) and np.allclose(ut.gia_that.values, gam.gia_that.values), \
    "Hai file không cùng thứ tự hàng — không ghép được"

D = pd.DataFrame({
    "thang":  ut.evaluation_month.values,
    "lag":    ut.requested_lag_minutes.values,
    "km":     ut.quote_distance.values,
    "gio":    ut.gio_vn.values,
    "y":      ut.gia_that.values,
    "hybrid": ut.hybrid_pred.values,
    "gam":    gam.hybrid_pred.values,
    "pers":   ut.persistence.values,
})
D = gan_nhom(D[D.lag == 5].reset_index(drop=True))
print(f"Test lag 5 phút: {len(D):,} chuyến · {D.thang.nunique()} tháng")
print(f"MAPE mốc Hybrid {mape(D.hybrid, D.y):.2%} · GAM {mape(D.gam, D.y):.2%}")

## 1. Chia tập dò tham số và tập đánh giá

In [ ]:
THANG_DO = "2026-01"
DO   = D[D.thang == THANG_DO].reset_index(drop=True)
GIA  = D[D.thang != THANG_DO].reset_index(drop=True)
print(f"Dò tham số : {THANG_DO} — {len(DO):,} chuyến")
print(f"Đánh giá   : {sorted(GIA.thang.unique())} — {len(GIA):,} chuyến")
print(f"\nTrên tập đánh giá: Hybrid {mape(GIA.hybrid, GIA.y):.2%} · GAM {mape(GIA.gam, GIA.y):.2%}")

## 2. Hàm trộn

In [ ]:
def alpha(d, d0, d1, amax):
    """Trọng số của GAM theo quãng đường — tuyến tính từng khúc, liên tục tại d0 và d1."""
    d = np.asarray(d, float)
    a = (d - d0) / max(d1 - d0, 1e-9)
    return np.clip(a, 0.0, 1.0) * amax

def tron(s, d0, d1, amax):
    a = alpha(s.km.values, d0, d1, amax)
    return (1 - a) * s.hybrid.values + a * s.gam.values

# minh hoa hinh dang ham tron
fig, ax = plt.subplots(figsize=(8, 3.2))
xx = np.linspace(0, 25, 400)
for d0, d1, am, mau in [(10, 18, 1.0, BLUE), (8, 20, 0.8, ORANGE), (12, 16, 1.0, GREEN)]:
    ax.plot(xx, alpha(xx, d0, d1, am), lw=2.2, color=mau,
            label=f"d₀={d0}, d₁={d1}, α_max={am}")
ax.set_xlabel("Quãng đường (km)"); ax.set_ylabel("Trọng số của GAM")
ax.set_title("Hình dạng hàm trộn — liên tục, không nhảy bậc", fontweight="bold")
ax.legend(frameon=False, fontsize=9.5)
fig.tight_layout(); plt.show()

## 3. Dò tham số trên tháng 2026-01

Tiêu chí chọn không phải "MAPE toàn tập thấp nhất" — GAM kém hơn trên toàn tập nên tiêu chí đó sẽ
luôn chọn `α_max = 0`, tức không trộn. Tiêu chí đúng phải phản ánh đúng câu hỏi của tuần:

> giảm sai số ở nhóm chuyến dài **mà không** làm toàn tập tệ đi quá `NGUONG_CHUNG` điểm.

Trong số các cấu hình thoả ràng buộc, chọn cấu hình giảm nhiều nhất ở nhóm `>12 km`.

In [ ]:
NGUONG_CHUNG = 0.05      # toan tap khong duoc xau di qua 0,05 diem MAPE

lưới = [(d0, d1, am)
        for d0 in [6, 8, 10, 12, 14]
        for d1 in [14, 16, 18, 20, 24]
        for am in [0.4, 0.6, 0.8, 1.0]
        if d1 > d0]
print(f"{len(lưới)} cấu hình")

moc_chung = mape(DO.hybrid, DO.y)
dai = DO[DO.km > 12]
moc_dai = mape(dai.hybrid, dai.y)

r = []
for d0, d1, am in lưới:
    p = tron(DO, d0, d1, am)
    pd_ = p[(DO.km > 12).values]
    r.append(dict(d0=d0, d1=d1, amax=am,
                  chung=mape(p, DO.y), dai=mape(pd_, dai.y)))
R = pd.DataFrame(r)
R["d_chung"] = (R.chung - moc_chung) * 100
R["d_dai"]   = (moc_dai - R.dai) * 100        # duong = tot len

hop_le = R[R.d_chung <= NGUONG_CHUNG]
print(f"{len(hop_le)}/{len(R)} cấu hình giữ được toàn tập trong ngưỡng {NGUONG_CHUNG} điểm")
if len(hop_le) == 0:
    raise SystemExit("Không cấu hình nào thoả ràng buộc — nới NGUONG_CHUNG hoặc bỏ hướng ghép")

tot = hop_le.loc[hop_le.d_dai.idxmax()]
D0, D1, AMAX = float(tot.d0), float(tot.d1), float(tot.amax)
print(f"\nChọn: d₀={D0:.0f} km · d₁={D1:.0f} km · α_max={AMAX}")
print(f"  trên tập dò — nhóm >12 km: −{tot.d_dai:.2f} điểm · toàn tập: {tot.d_chung:+.3f} điểm")

In [ ]:
# ═════════ HÌNH GG1 — ban do do tham so ═════════
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(R.d_chung, R.d_dai, s=26, c=R.amax, cmap="viridis", alpha=.8)
ax[0].axvline(NGUONG_CHUNG, color=RED, ls="--", lw=1.6,
              label=f"ngưỡng toàn tập {NGUONG_CHUNG} điểm")
ax[0].scatter([tot.d_chung], [tot.d_dai], s=200, marker="*", color=RED,
              zorder=5, label="cấu hình chọn")
ax[0].axhline(0, color=MUT, lw=1); ax[0].axvline(0, color=MUT, lw=1)
ax[0].set_xlabel("Toàn tập xấu đi (điểm) →")
ax[0].set_ylabel("Nhóm >12 km tốt lên (điểm) →")
ax[0].set_title("Mỗi chấm một cấu hình · màu = α_max", fontweight="bold")
ax[0].legend(frameon=False, fontsize=9.5)

piv = R[R.amax == AMAX].pivot(index="d0", columns="d1", values="d_dai")
im = ax[1].imshow(piv.values, cmap="YlGn", aspect="auto", origin="lower")
ax[1].set_xticks(range(len(piv.columns))); ax[1].set_xticklabels(piv.columns)
ax[1].set_yticks(range(len(piv.index)));   ax[1].set_yticklabels(piv.index)
ax[1].set_xlabel("d₁ (km)"); ax[1].set_ylabel("d₀ (km)")
ax[1].set_title(f"Nhóm >12 km tốt lên bao nhiêu điểm · α_max={AMAX}", fontweight="bold")
ax[1].grid(False)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if not np.isnan(v):
            ax[1].text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8.5)
fig.colorbar(im, ax=ax[1], fraction=.04)

fig.suptitle(f"GG1 — Dò tham số trộn trên tháng {THANG_DO} (tập đánh giá chưa đụng tới)",
             fontweight="bold", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(HINH / "GG1_do_tham_so.png")
plt.show()

## 4. Đánh giá trên hai tháng chưa đụng tới

Đây là con số đưa vào báo cáo.

In [ ]:
GIA = GIA.copy()
GIA["ghep"] = tron(GIA, D0, D1, AMAX)

print(f"Tập đánh giá: {sorted(GIA.thang.unique())} — {len(GIA):,} chuyến")
print(f"  Hybrid : {mape(GIA.hybrid, GIA.y):.2%}")
print(f"  GAM    : {mape(GIA.gam,    GIA.y):.2%}")
print(f"  Ghép   : {mape(GIA.ghep,   GIA.y):.2%}")

BG = bang_theo_nhom(GIA, "hybrid", "ghep", ten_moc="Hybrid", ten_moi="Ghép")
in_bang(BG, ["Hybrid", "Ghép"])

In [ ]:
# ═════════ HÌNH GG2 — ba model theo quang duong ═════════
BIEN = [0, 2, 5, 8, 10, 12, 14, 16, 20, np.inf]
GIA["kb"] = pd.cut(GIA.km, BIEN)
r = []
for g, s in GIA.groupby("kb", observed=True):
    if len(s) < 50:
        continue
    r.append(dict(giua=(g.left + min(g.right, 22))/2, n=len(s),
                  Hybrid=mape(s.hybrid, s.y)*100, GAM=mape(s.gam, s.y)*100,
                  Ghép=mape(s.ghep, s.y)*100))
Q = pd.DataFrame(r)

fig, ax = plt.subplots(1, 2, figsize=(14.5, 5))
for ten, mau, kt in [("Hybrid", BLUE, "o-"), ("GAM", ORANGE, "s--"), ("Ghép", GREEN, "^-")]:
    ax[0].plot(Q.giua, Q[ten], kt, color=mau, lw=2.3, ms=7, label=ten)
ax[0].axvline(D0, color=MUT, ls=":", lw=1.4)
ax[0].axvline(D1, color=MUT, ls=":", lw=1.4)
ax[0].annotate(f"d₀={D0:.0f}", (D0, ax[0].get_ylim()[1]), fontsize=9, color=MUT,
               ha="center", va="top")
ax[0].annotate(f"d₁={D1:.0f}", (D1, ax[0].get_ylim()[1]), fontsize=9, color=MUT,
               ha="center", va="top")
ax[0].set_xlabel("Quãng đường (km)"); ax[0].set_ylabel("MAPE (%)")
ax[0].set_title("Sai số ba model theo quãng đường", fontweight="bold")
ax[0].legend(frameon=False)

xx = np.linspace(0, 24, 300)
ax[1].plot(xx, alpha(xx, D0, D1, AMAX), lw=2.6, color=GREEN)
ax[1].fill_between(xx, 0, alpha(xx, D0, D1, AMAX), color=GREEN, alpha=.14)
ax[1].set_xlabel("Quãng đường (km)"); ax[1].set_ylabel("Trọng số của GAM")
ax[1].set_ylim(-.03, 1.05)
ax[1].set_title(f"Hàm trộn đã chọn · d₀={D0:.0f}, d₁={D1:.0f}, α_max={AMAX}",
                fontweight="bold")

fig.suptitle("GG2 — Ghép GAM–GBM theo quãng đường, đo trên hai tháng chưa đụng tới",
             fontweight="bold", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(HINH / "GG2_ghep_theo_km.png")
plt.show()

## 4b. Trộn ở mức nhánh giá cơ bản

`05_GAM_CHI_TIET` cho thấy lợi thế của GAM ở chuyến dài nằm **hoàn toàn** ở nhánh giá cơ bản
(+2,30 điểm ở nhóm `>15 km`), còn nhánh hệ số nhân thì GAM kém đều khoảng 0,5 điểm ở mọi nhóm.

Vậy trộn giá cuối là kéo theo cả nhánh yếu một cách vô ích. Cách đúng hơn: **chỉ trộn nhánh giá cơ
bản, luôn giữ hệ số nhân của GBM.**

$$\hat p = \big[(1-\alpha(d)) \cdot \hat b_{GBM} + \alpha(d) \cdot \hat b_{GAM}\big] \times \hat m_{GBM}$$

Vẫn đúng tinh thần *"không xây thêm kiến trúc phức tạp"* — cùng một hàm trọng số, chỉ khác chỗ áp
dụng.

In [ ]:
cb = pd.read_parquet(EVAL / "pred_gia_co_ban.parquet")
hs = pd.read_parquet(EVAL / "pred_heso.parquet", columns=["pred", "algo"])
m5 = (ut.requested_lag_minutes == 5).values      # cung mask da dung khi dung D

D2 = D.copy()
D2["b_gbm"] = cb[cb.algo == "HistGB"].base_pred.values[m5]
D2["h_gbm"] = hs[hs.algo == "HistGB"].pred.values[m5]
D2["b_gam"] = gam.base_pred.values[m5]
assert np.allclose(D2.b_gbm * D2.h_gbm, D2.hybrid, rtol=1e-6), \
    "hai nhánh nhân lại không ra đúng dự đoán Hybrid — thứ tự hàng lệch"
# DO/GIA o muc 1 da reset_index nen KHONG join theo index duoc — loc lai tu D2
DO2  = D2[D2.thang == THANG_DO].reset_index(drop=True)
GIA2 = D2[D2.thang != THANG_DO].reset_index(drop=True)
assert len(DO2) == len(DO) and len(GIA2) == len(GIA)
assert np.allclose(GIA2.y.values, GIA.y.values), "lọc lại không khớp thứ tự"
GIA2["ghep"] = tron(GIA2, D0, D1, AMAX)      # tinh lai, giong het muc 4

def tron_nhanh(s, d0, d1, amax):
    a = alpha(s.km.values, d0, d1, amax)
    return ((1 - a) * s.b_gbm.values + a * s.b_gam.values) * s.h_gbm.values

# do lai tham so RIENG cho cach tron nay, van tren thang do
moc_chung2 = mape(DO2.hybrid, DO2.y)
dai2 = DO2[DO2.km > 12]
moc_dai2 = mape(dai2.hybrid, dai2.y)
r = []
for d0, d1, am in lưới:
    p = tron_nhanh(DO2, d0, d1, am)
    r.append(dict(d0=d0, d1=d1, amax=am, chung=mape(p, DO2.y),
                  dai=mape(p[(DO2.km > 12).values], dai2.y)))
R2 = pd.DataFrame(r)
R2["d_chung"] = (R2.chung - moc_chung2) * 100
R2["d_dai"]   = (moc_dai2 - R2.dai) * 100
hop_le2 = R2[R2.d_chung <= NGUONG_CHUNG]
tot2 = hop_le2.loc[hop_le2.d_dai.idxmax()]
E0, E1, EMAX = float(tot2.d0), float(tot2.d1), float(tot2.amax)
print(f"Trộn nhánh cơ bản — chọn: d₀={E0:.0f} · d₁={E1:.0f} · α_max={EMAX}")

GIA2["ghep_nhanh"] = tron_nhanh(GIA2, E0, E1, EMAX)
print(f"\nTrên tập đánh giá {sorted(GIA2.thang.unique())}:")
print(f"  Hybrid GBM        : {mape(GIA2.hybrid, GIA2.y):.2%}")
print(f"  Ghép giá cuối     : {mape(GIA2.ghep, GIA2.y):.2%}")
print(f"  Ghép nhánh cơ bản : {mape(GIA2.ghep_nhanh, GIA2.y):.2%}")

In [ ]:
BN = bang_theo_nhom(GIA2, "hybrid", "ghep_nhanh", ten_moc="Hybrid", ten_moi="Ghép nhánh")
in_bang(BN, ["Hybrid", "Ghép nhánh"])

In [ ]:
# So truc tiep hai cach ghep tren nhom chuyen dai
print("So hai cách ghép với nhau (dương = ghép nhánh cơ bản tốt hơn):")
for g in [">15", "12–15", "TOÀN TẬP"]:
    s = GIA2 if g == "TOÀN TẬP" else GIA2[GIA2.kmb.astype(str) == g]
    if len(s) < 30:
        continue
    m, lo, hi = boot_hieu(s.ghep, s.ghep_nhanh, s.y)
    ro = "✔ khác 0" if (lo > 0 or hi < 0) else "không phân biệt"
    print(f"  {g:>10} n={len(s):>7,}  {m*100:+6.2f} điểm "
          f"[{lo*100:+6.2f}, {hi*100:+6.2f}]  {ro}")

## 5. Kiểm tra chuyển tiếp có mượt không

Khách đi 14,9 km và 15,1 km không nên nhận giá lệch hẳn nhau.

**Đo cái gì.** Không so `ghép / giá thật` ở hai bên mốc — chuyến hai bên mốc vốn khác nhau, nên con
số đó lẫn cả sự khác biệt giữa các chuyến với ảnh hưởng của việc trộn, và sẽ báo động giả ở vùng
thưa. Cách đúng là so **`ghép / Hybrid`**: cùng một chuyến làm mốc quy chiếu, nên phần chênh còn lại
chỉ do hàm trộn.

In [ ]:
def nhay_bac(mocc, dai=1.0):
    trai = GIA[(GIA.km >= mocc - dai) & (GIA.km < mocc)]
    phai = GIA[(GIA.km >= mocc) & (GIA.km < mocc + dai)]
    if len(trai) < 20 or len(phai) < 20:
        return None
    # ghep/hybrid = 1 − α + α·(gam/hybrid)
    t = (trai.ghep / trai.hybrid).mean(); p = (phai.ghep / phai.hybrid).mean()
    at = alpha(trai.km.values, D0, D1, AMAX).mean()
    ap = alpha(phai.km.values, D0, D1, AMAX).mean()
    # Ky vong: chenh CHI do α doi, giu nguyen ty le gam/hybrid quanh moc
    ty = pd.concat([trai, phai]).eval("gam / hybrid").mean()
    ky_vong = (ap - at) * (ty - 1)
    return dict(mốc=mocc, n=len(trai)+len(phai),
                α_trái=at, α_phải=ap,
                quan_sát=(p - t)*100, kỳ_vọng=ky_vong*100,
                dư=((p - t) - ky_vong)*100)

r = [x for x in (nhay_bac(m) for m in [D0, (D0+D1)/2, D1]) if x]
NB_ = pd.DataFrame(r)
if len(NB_):
    print("Chênh lệch tỷ lệ (ghép / Hybrid) qua các mốc, đơn vị điểm %:")
    print(NB_.round(4).to_string(index=False))
    print("\n  quan_sát : chênh thực tế giữa hai bên mốc")
    print("  kỳ_vọng  : phần giải thích được bằng riêng thay đổi của α")
    print("  dư       : phần CÒN LẠI — đây mới là dấu hiệu bất thường nếu lớn")
    print("\nĐọc đúng: hàm trộn LIÊN TỤC theo cấu tạo nên không tồn tại bậc nhảy để tìm.")
    print("Phần chênh còn lại phản ánh việc GAM và GBM vốn lệch nhau nhiều dần theo quãng")
    print("đường — đúng là điều khiến việc ghép có ích. Hình dưới cho thấy nó biến thiên")
    print("mượt chứ không gãy khúc.")
else:
    print("Không đủ chuyến quanh các mốc để kiểm tra — nhóm quãng đường dài rất thưa.")

# ═════════ HÌNH GG3 — ghep lech khoi Hybrid bao nhieu, theo km ═════════
BI = np.arange(0, 22, 1.0)
GIA["kb1"] = pd.cut(GIA.km, BI)
r = [dict(giua=g.mid, n=len(s), ty=(s.ghep / s.hybrid).mean(),
          a=alpha(s.km.values, D0, D1, AMAX).mean())
     for g, s in GIA.groupby("kb1", observed=True) if len(s) >= 30]
C = pd.DataFrame(r)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(C.giua, (C.ty - 1)*100, "o-", color=GREEN, lw=2.4, ms=6,
        label="giá ghép lệch khỏi Hybrid (%)")
ax.axhline(0, color=INK, lw=1.2)
ax.axvline(D0, color=MUT, ls=":", lw=1.4); ax.axvline(D1, color=MUT, ls=":", lw=1.4)
a2 = ax.twinx()
a2.plot(C.giua, C.a, "--", color=ORANGE, lw=1.8, label="trọng số GAM (α)")
a2.set_ylabel("α", color=ORANGE); a2.set_ylim(-.03, 1.05); a2.grid(False)
ax.set_xlabel("Quãng đường (km)")
ax.set_ylabel("Lệch so với Hybrid (%)")
ax.set_title("GG3 — Ghép làm giá đổi bao nhiêu so với Hybrid\n"
             "Đường liền biến thiên mượt, không gãy khúc tại d₀ hay d₁",
             fontweight="bold", fontsize=12.5)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, frameon=False, loc="lower left")
fig.tight_layout()
fig.savefig(HINH / "GG3_lech_theo_km.png")
plt.show()

## 6. Lưu kết quả

In [ ]:
BG.to_csv(KQ / "B2_ghep_vs_hybrid.csv", index=False)
Q.to_csv(KQ / "B2_theo_quang_duong.csv", index=False)
R.to_csv(KQ / "B2_luoi_tham_so.csv", index=False)
json.dump({"d0": D0, "d1": D1, "amax": AMAX, "thang_do": THANG_DO,
           "thang_danh_gia": sorted(GIA.thang.unique().tolist()),
           "mape_hybrid": mape(GIA.hybrid, GIA.y),
           "mape_gam": mape(GIA.gam, GIA.y),
           "mape_ghep": mape(GIA.ghep, GIA.y)},
          open(KQ / "B2_cau_hinh.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Đã lưu vào", KQ.resolve())

## 7. Kết luận cần điền

1. Trên hai tháng chưa đụng tới, ghép có tốt hơn Hybrid ở nhóm `>15 km` không, và CI có loại trừ 0
   không?
2. Toàn tập xấu đi bao nhiêu điểm?
3. Chuyển tiếp có mượt không?

> Nếu ghép chỉ tốt hơn trên tập dò tham số mà không tốt hơn trên tập đánh giá, đó là **overfit tham
> số trộn** — báo cáo đúng như vậy và bỏ hướng này.